# SOXX 成分股下跌集中度：身份来源门

冻结 Stage1 衍生假设：比较原始 N-PORT 与旧 Qlib 摄取表的证券身份和披露权重，不计算收益、特征或账户。

In [1]:
import collections, csv, glob, hashlib, json, pathlib, re, xml.etree.ElementTree as ET
from decimal import Decimal
root=pathlib.Path.cwd();card=root/'research/stage1_soxx_constituent_downside_identity_preflight_20261023'
plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert sha(plan['inputs']['cached_crosswalk'])==plan['inputs']['cached_crosswalk_sha256']
assert sha(plan['inputs']['qlib_price_manifest'])==plan['inputs']['qlib_price_manifest_sha256']
manifest=json.loads((root/plan['inputs']['original_xml_sha_manifest']).read_text())['original_xml_sha256']
patterns=['data/snapshots/stage1_soxx_2019_nport_source_specimen_20261023/*.xml','data/snapshots/stage1_soxx_2019_2022_nport_original_xml_20261023/*.xml','data/snapshots/stage1_soxx_2023_2026_nport_original_source_20261023/*.xml']
files=sorted(p for pattern in patterns for p in glob.glob(pattern));assert len(files)==28
original={};reports={}
for path in files:
 name=pathlib.Path(path).name;assert sha(path)==manifest[name]
 accession='0001752724-19-177706' if name=='primary_doc.xml' else re.search(r'\d{10}-\d{2}-\d{6}',name).group()
 tree=ET.parse(path).getroot();period=next(x.text for x in tree.iter() if x.tag.rsplit('}',1)[-1]=='repPdDate');reports[accession]=period
 members=[]
 for node in tree.iter():
  if node.tag.rsplit('}',1)[-1]!='invstOrSec':continue
  fields={x.tag.rsplit('}',1)[-1]:x for x in node}
  value=lambda k:(fields[k].text or '').strip() if k in fields else ''
  if value('assetCat')!='EC' or value('units')!='NS':continue
  ids=fields.get('identifiers');isins=[] if ids is None else [x.attrib.get('value','') for x in ids if x.tag.rsplit('}',1)[-1]=='isin']
  assert len(isins)==1 and isins[0]
  members.append({'isin':isins[0],'cusip':value('cusip'),'name':value('name'),'value':Decimal(value('valUSD')),'pct':Decimal(value('pctVal'))})
 assert len(members)==30 and len({x['isin'] for x in members})==30
 for member in members:original[(accession,member['isin'])]=member
cache=list(csv.DictReader((root/plan['inputs']['cached_crosswalk']).open()));assert len(cache)==840
cached={};isin_tickers=collections.defaultdict(set);ticker_isins=collections.defaultdict(set)
for x in cache:
 key=(x['accession'],x['isin']);assert key not in cached;cached[key]=x
 isin_tickers[x['isin']].add(x['ticker']);ticker_isins[x['ticker']].add(x['isin'])
missing_original=sorted(set(original)-set(cached));extra_cached=sorted(set(cached)-set(original))
field_mismatches=[]
for key in set(original)&set(cached):
 a=original[key];b=cached[key]
 for field,observed,expected in [('report_period',b['report_date'],reports[key[0]]),('cusip',b['cusip'],a['cusip']),('name',b['name'],a['name']),('market_value_usd',Decimal(b['market_value_usd']),a['value']),('weight_pct',Decimal(b['weight_pct']),a['pct'])]:
  if observed!=expected:field_mismatches.append({'accession':key[0],'isin':key[1],'field':field,'cached':str(observed),'original':str(expected)})
price_manifest=json.loads((root/plan['inputs']['qlib_price_manifest']).read_text())
provider=set(price_manifest['instruments'])
uncached_tickers=sorted(set(ticker_isins)-provider)
result={'decision':'original_NPORT_crosswalk_identity_audit','plan_sha256':sha(card/'plan.json'),'original_reports':len(files),'original_equity_rows':len(original),'cached_rows':len(cache),'report_periods':len(set(reports.values())),'missing_original_rows':missing_original,'extra_cached_rows':extra_cached,'field_mismatches':field_mismatches,'unique_original_ISINs':len(isin_tickers),'unique_cached_tickers':len(ticker_isins),'ISIN_to_multiple_tickers':{k:sorted(v) for k,v in isin_tickers.items() if len(v)>1},'ticker_to_multiple_ISINs':{k:sorted(v) for k,v in ticker_isins.items() if len(v)>1},'cached_tickers_absent_from_Qlib_price_provider':uncached_tickers,'Qlib_price_provider_calendar_start':price_manifest.get('calendar_start'),'Qlib_price_provider_calendar_end':price_manifest.get('calendar_end'),'Stage1_held_last_signal':'2026-10-07','price_provider_reaches_held_end':price_manifest.get('calendar_end','')>='2026-10-07','placeholder_CUSIP_rows':sum(x['cusip'] in ('000000000','N/A','') for x in original.values()),'source_crosswalk_exact':not missing_original and not extra_cached and not field_mismatches,'feature_source_qualified':False,'market_price_reads':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k not in ('missing_original_rows','extra_cached_rows','field_mismatches','ISIN_to_multiple_tickers','ticker_to_multiple_ISINs')},ensure_ascii=False,indent=2))
print('mismatch counts',len(missing_original),len(extra_cached),len(field_mismatches),'mapping conflicts',len(result['ISIN_to_multiple_tickers']),len(result['ticker_to_multiple_ISINs']))


{
  "decision": "original_NPORT_crosswalk_identity_audit",
  "plan_sha256": "c4be5381a5b4aa43e266eee473e155309a996dc6e18467c2b3edc1c83bb933b3",
  "original_reports": 28,
  "original_equity_rows": 840,
  "cached_rows": 840,
  "report_periods": 27,
  "unique_original_ISINs": 50,
  "unique_cached_tickers": 47,
  "cached_tickers_absent_from_Qlib_price_provider": [
    "BRKS",
    "CCMP",
    "CREE",
    "IPHI",
    "MXIM",
    "WOLF",
    "XLNX"
  ],
  "Qlib_price_provider_calendar_start": "2007-02-01",
  "Qlib_price_provider_calendar_end": "2026-07-02",
  "Stage1_held_last_signal": "2026-10-07",
  "price_provider_reaches_held_end": false,
  "placeholder_CUSIP_rows": 71,
  "source_crosswalk_exact": true,
  "feature_source_qualified": false,
  "market_price_reads": 0,
  "SOXX_outcome_reads": 0,
  "model_fits": 0,
  "native_accounts": 0
}
mismatch counts 0 0 0 mapping conflicts 0 3


In [2]:
import collections, csv, hashlib, json, pathlib, statistics
card=pathlib.Path('research/stage1_soxx_constituent_downside_identity_preflight_20261023')
plan=json.loads((card/'triage_plan.json').read_text())
assert hashlib.sha256(pathlib.Path(plan['source']).read_bytes()).hexdigest()==plan['source_sha256']
base=json.loads((card/'result.json').read_text())
rows=list(csv.DictReader(open(plan['source'])))
assert len(rows)==840 and base['source_crosswalk_exact']
missing=set(base['cached_tickers_absent_from_Qlib_price_provider'])
by_report=collections.defaultdict(list);reuse=collections.defaultdict(lambda:collections.defaultdict(set))
for r in rows:
 by_report[(r['report_date'],r['accession'])].append(r)
 if r['ticker'] in base['ticker_to_multiple_ISINs']:
  reuse[r['ticker']][r['isin']].add(r['report_date'])
report_weights=[]
for (period,accession),members in sorted(by_report.items()):
 absent=sum(float(r['weight_pct']) for r in members if r['ticker'] in missing)
 report_weights.append({'report_period':period,'accession':accession,'absent_provider_ticker_pct_net_assets':absent,'absent_tickers':sorted({r['ticker'] for r in members if r['ticker'] in missing})})
result={'plan_sha256':hashlib.sha256((card/'triage_plan.json').read_bytes()).hexdigest(),'reports':len(report_weights),'absent_weight_pct_min':min(x['absent_provider_ticker_pct_net_assets'] for x in report_weights),'absent_weight_pct_median':statistics.median(x['absent_provider_ticker_pct_net_assets'] for x in report_weights),'absent_weight_pct_max':max(x['absent_provider_ticker_pct_net_assets'] for x in report_weights),'reports_with_absent_ticker':sum(x['absent_provider_ticker_pct_net_assets']>0 for x in report_weights),'highest_absent_weight_reports':sorted(report_weights,key=lambda x:x['absent_provider_ticker_pct_net_assets'],reverse=True)[:5],'ticker_reuse_periods':{ticker:{isin:sorted(periods) for isin,periods in ids.items()} for ticker,ids in reuse.items()},'market_price_reads':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'triage_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k!='ticker_reuse_periods'},ensure_ascii=False,indent=2))


{
  "plan_sha256": "11d3a25ec747b30e5eebacd9aced609b5d51cf08666425f4521007e2c0054214",
  "reports": 28,
  "absent_weight_pct_min": 0,
  "absent_weight_pct_median": 1.0274275865535,
  "absent_weight_pct_max": 7.678446417541,
  "reports_with_absent_ticker": 20,
  "highest_absent_weight_reports": [
    {
      "report_period": "2019-09-30",
      "accession": "0001752724-19-177706",
      "absent_provider_ticker_pct_net_assets": 7.678446417541,
      "absent_tickers": [
        "CCMP",
        "CREE",
        "MXIM",
        "XLNX"
      ]
    },
    {
      "report_period": "2019-12-31",
      "accession": "0001752724-20-038669",
      "absent_provider_ticker_pct_net_assets": 7.014987162045,
      "absent_tickers": [
        "CCMP",
        "CREE",
        "MXIM",
        "XLNX"
      ]
    },
    {
      "report_period": "2020-03-31",
      "accession": "0001752724-20-112013",
      "absent_provider_ticker_pct_net_assets": 6.803638760428,
      "absent_tickers": [
        "CCMP",
      

## 中文结论

原始 28 份 N-PORT 的 840 行与缓存表按 accession＋ISIN 精确对应，名称、CUSIP、持仓价值和原始权重均无差异。50 个 ISIN 对应 47 个 ticker；CCMP、MRVL、LRCX 各跨两个 ISIN。七个历史代码在所选 Qlib 价格库中不存在，20/28 份报告有缺价，最大原始净资产权重 7.68%。价格库止于 2026-07-02，早于 Stage1 评价末日 2026-10-07。

只通过身份交叉表门，价格/PIT/完整评价期仍未合格。零价格或 SOXX 结果读取、零拟合、零账户，不晋级。
此前同机制 2019–2022 原生 Qlib 价格门已得到完整 30 股窗口 0/780；本卡不重跑价格检验或拟合，只扩展 2023–2026 原件身份审计。